In [12]:
import pulp as pl

#Datos
capital = 20600000
rates = {
    "CDT_A": 0.0385,  # 3 meses
    "CDT_B": 0.039,  # 6 meses
    "CDT_C": 0.0395,  # 9 meses
    "CDT_D": 0.04   # 12 meses
}

short = ["CDT_A", "CDT_B"]
long = ["CDT_C", "CDT_D"]

def solve_case(case_id, constraints):
    # Modelo
    model = pl.LpProblem(f"Caso_{case_id}", pl.LpMaximize)
    
    # Variables de decisión
    x = {name: pl.LpVariable(name, lowBound=0) for name in rates}
    
    # Restricción de presupuesto
    model += pl.lpSum(x[i] for i in rates) == capital
    
    # Restricciones adicionales según el caso
    for cons in constraints:
        model += cons(x)
    
    # Función objetivo: maximizar retorno
    model += pl.lpSum(rates[i] * x[i] for i in rates)
    
    # Resolver
    model.solve(pl.PULP_CBC_CMD(msg=False))
    
    # Resultados
    print(f"\n--- Caso {case_id} ---")
    for i in x:
        print(f"{i}: {x[i].value():,.2f}")
    print("Retorno total:", pl.value(model.objective))

# Caso 1: 100% corto plazo
solve_case(1, [
    lambda x: pl.lpSum(x[i] for i in short) == capital,
    lambda x: pl.lpSum(x[i] for i in long) == 0
])

# Caso 2: >=50% corto, <=50% largo
solve_case(2, [
    lambda x: pl.lpSum(x[i] for i in short) >= 0.5 * capital,
    lambda x: pl.lpSum(x[i] for i in long) <= 0.5 * capital
])

# Caso 3: <=50% corto, >=50% largo
solve_case(3, [
    lambda x: pl.lpSum(x[i] for i in short) <= 0.5 * capital,
    lambda x: pl.lpSum(x[i] for i in long) >= 0.5 * capital
])

# Caso 4: exactamente 25% corto, 25% largo cada uno
solve_case(4, [
    lambda x: x["CDT_A"] == 0.25 * capital,
    lambda x: x["CDT_B"] == 0.25 * capital,
    lambda x: x["CDT_C"] == 0.25 * capital,
    lambda x: x["CDT_D"] == 0.25 * capital
])

# Caso 5: <=25% corto, >=25% largo
solve_case(5, [
    lambda x: pl.lpSum(x[i] for i in short) <= 0.25 * capital,
    lambda x: pl.lpSum(x[i] for i in long) >= 0.25 * capital
])



--- Caso 1 ---
CDT_A: 0.00
CDT_B: 20,600,000.00
CDT_C: 0.00
CDT_D: 0.00
Retorno total: 803400.0

--- Caso 2 ---
CDT_A: 0.00
CDT_B: 10,300,000.00
CDT_C: 0.00
CDT_D: 10,300,000.00
Retorno total: 813700.0

--- Caso 3 ---
CDT_A: 0.00
CDT_B: 0.00
CDT_C: 0.00
CDT_D: 20,600,000.00
Retorno total: 824000.0

--- Caso 4 ---
CDT_A: 5,150,000.00
CDT_B: 5,150,000.00
CDT_C: 5,150,000.00
CDT_D: 5,150,000.00
Retorno total: 808550.0

--- Caso 5 ---
CDT_A: 0.00
CDT_B: 0.00
CDT_C: 0.00
CDT_D: 20,600,000.00
Retorno total: 824000.0
